# Parameter Optimization with Grid Search

This notebook demonstrates:
1. Grid search parameter optimization
2. Comparing performance across parameter combinations
3. Visualizing optimization results
4. Finding optimal RSI strategy parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from itertools import product
import kimsfinance_core
from kimsfinance.strategies import RSIStrategy
from kimsfinance.visualization import print_performance_summary

print(f"kimsfinance_core v{kimsfinance_core.__version__}")

## 1. Generate Test Data

In [ ]:
def generate_sample_data(n=1000, seed=42):
    np.random.seed(seed)
    timestamps = np.arange(n, dtype=np.int64) * 60
    
    # Trending market with noise
    base = np.linspace(100.0, 180.0, n)
    noise = np.random.randn(n).cumsum() * 3
    close = base + noise
    
    open_prices = close + np.random.randn(n) * 0.5
    high = np.maximum(open_prices, close) + np.abs(np.random.randn(n) * 2)
    low = np.minimum(open_prices, close) - np.abs(np.random.randn(n) * 2)
    volume = np.random.uniform(1000, 10000, n)
    
    return timestamps, open_prices, high, low, close, volume

timestamps, open_p, high, low, close, volume = generate_sample_data(2000)
print(f"Generated {len(close)} candles for optimization")

## 2. Define Parameter Grid

In [ ]:
# Parameter ranges to test
periods = [7, 10, 14, 20, 30]
buy_thresholds = [20, 25, 30, 35, 40]
sell_thresholds = [60, 65, 70, 75, 80]

# Generate all combinations
param_grid = list(product(periods, buy_thresholds, sell_thresholds))

print(f"Total parameter combinations: {len(param_grid)}")
print(f"Example combinations:")
for params in param_grid[:5]:
    print(f"  Period={params[0]}, Buy={params[1]}, Sell={params[2]}")

## 3. Run Grid Search

In [ ]:
results = []

for i, (period, buy_thresh, sell_thresh) in enumerate(param_grid):
    if i % 10 == 0:
        print(f"Testing combination {i+1}/{len(param_grid)}...")
    
    # Create strategy with these parameters
    strategy = RSIStrategy(
        period=period,
        buy_threshold=buy_thresh,
        sell_threshold=sell_thresh
    )
    
    # Run backtest
    try:
        result = kimsfinance_core.run_backtest(
            high=high,
            low=low,
            close=close,
            open_prices=open_p,
            volume=volume,
            timestamps=timestamps,
            strategy=strategy,
            initial_capital=10000.0,
            trading_fee=0.001,
            slippage=0.0005,
            use_gpu=False
        )
        
        results.append({
            'period': period,
            'buy_threshold': buy_thresh,
            'sell_threshold': sell_thresh,
            'sharpe_ratio': result['sharpe_ratio'],
            'total_return': result['total_return'],
            'max_drawdown': result['max_drawdown'],
            'num_trades': result['num_trades'],
            'win_rate': result['win_rate'],
            'profit_factor': result['profit_factor']
        })
    except Exception as e:
        print(f"Error with params {period}, {buy_thresh}, {sell_thresh}: {e}")

results_df = pd.DataFrame(results)
print(f"\nCompleted {len(results_df)} backtests")

## 4. Analyze Results

### Top Performing Parameters

In [ ]:
# Sort by Sharpe ratio
top_10 = results_df.nlargest(10, 'sharpe_ratio')
print("Top 10 parameter combinations by Sharpe Ratio:")
print(top_10)

### Visualize Parameter Impact

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Sharpe vs Period
period_stats = results_df.groupby('period')['sharpe_ratio'].agg(['mean', 'std'])
axes[0,0].errorbar(period_stats.index, period_stats['mean'], yerr=period_stats['std'], marker='o', capsize=5)
axes[0,0].set_title('Sharpe Ratio vs RSI Period', fontsize=12, fontweight='bold')
axes[0,0].set_xlabel('Period')
axes[0,0].set_ylabel('Sharpe Ratio')
axes[0,0].grid(True, alpha=0.3)

# 2. Return vs Buy Threshold
buy_stats = results_df.groupby('buy_threshold')['total_return'].agg(['mean', 'std'])
axes[0,1].errorbar(buy_stats.index, buy_stats['mean'], yerr=buy_stats['std'], marker='o', capsize=5, color='green')
axes[0,1].set_title('Total Return vs Buy Threshold', fontsize=12, fontweight='bold')
axes[0,1].set_xlabel('Buy Threshold')
axes[0,1].set_ylabel('Total Return (%)')
axes[0,1].grid(True, alpha=0.3)

# 3. Drawdown vs Sell Threshold
sell_stats = results_df.groupby('sell_threshold')['max_drawdown'].agg(['mean', 'std'])
axes[1,0].errorbar(sell_stats.index, sell_stats['mean'], yerr=sell_stats['std'], marker='o', capsize=5, color='red')
axes[1,0].set_title('Max Drawdown vs Sell Threshold', fontsize=12, fontweight='bold')
axes[1,0].set_xlabel('Sell Threshold')
axes[1,0].set_ylabel('Max Drawdown (%)')
axes[1,0].grid(True, alpha=0.3)

# 4. Scatter: Return vs Risk
axes[1,1].scatter(results_df['max_drawdown'], results_df['total_return'], 
                 c=results_df['sharpe_ratio'], cmap='viridis', alpha=0.6, s=50)
axes[1,1].set_title('Return vs Risk (colored by Sharpe)', fontsize=12, fontweight='bold')
axes[1,1].set_xlabel('Max Drawdown (%)')
axes[1,1].set_ylabel('Total Return (%)')
axes[1,1].grid(True, alpha=0.3)
plt.colorbar(axes[1,1].collections[0], ax=axes[1,1], label='Sharpe Ratio')

plt.tight_layout()
plt.show()

## 5. Test Optimal Parameters

In [ ]:
# Get best parameters
best = results_df.loc[results_df['sharpe_ratio'].idxmax()]

print("Optimal parameters:")
print(f"  Period: {best['period']}")
print(f"  Buy threshold: {best['buy_threshold']}")
print(f"  Sell threshold: {best['sell_threshold']}")
print(f"\nExpected performance:")
print(f"  Sharpe ratio: {best['sharpe_ratio']:.3f}")
print(f"  Total return: {best['total_return']:.2f}%")
print(f"  Max drawdown: {best['max_drawdown']:.2f}%")

# Run backtest with optimal parameters
optimal_strategy = RSIStrategy(
    period=int(best['period']),
    buy_threshold=best['buy_threshold'],
    sell_threshold=best['sell_threshold']
)

optimal_result = kimsfinance_core.run_backtest(
    high=high,
    low=low,
    close=close,
    open_prices=open_p,
    volume=volume,
    timestamps=timestamps,
    strategy=optimal_strategy,
    initial_capital=10000.0,
    trading_fee=0.001,
    slippage=0.0005,
    use_gpu=False
)

print_performance_summary(optimal_result)

## 6. Next Steps

- Try genetic algorithms for faster optimization (see `03_genetic_optimization.ipynb`)
- Test on out-of-sample data to avoid overfitting
- Combine multiple indicators for robustness
- Use walk-forward optimization for production strategies